# CT Visualization in Google Colab

This notebook mounts Google Drive, loads all `CT*.xlsx` files from `/content/drive/MyDrive/CT/`, and creates combined plots for coulometric titration data.

## 1) Mount Google Drive
Run this cell first so Colab can access your Excel files in Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2) Import libraries and find Excel files
This cell searches `/content/drive/MyDrive/CT/` for files named like `CTSTF50.xlsx` or `CTSr3Fe2O7.xlsx`.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

data_dir = Path('/content/drive/MyDrive/CTEX/')
excel_files = sorted(data_dir.glob('CT*.xlsx'))

print(f'Found {len(excel_files)} Excel file(s) in {data_dir}')

if not excel_files:
    raise FileNotFoundError(
        'No files matching CT*.xlsx were found. '
        'Please check that your Google Drive folder path is correct: '
        '/content/drive/MyDrive/CT/'
    )

## 3) Load and clean data from each file
The notebook reads the first sheet in each Excel file and uses Excel column positions:
- Column H (index 7): `log p(O2)`
- Column I (index 8): `delta at atm - delta`
- Column J (index 9): `delta`

Rows with non-numeric or blank values in required columns are dropped.

In [ ]:
samples = []

for file_path in excel_files:
    try:
        df = pd.read_excel(file_path, sheet_name=0, header=None)
    except Exception as exc:
        print(f'Warning: Could not read {file_path.name}. Skipping. Error: {exc}')
        continue

    needed = df.iloc[:, [7, 8, 9]].copy()
    needed.columns = ['log_pO2', 'delta_atm_minus_delta', 'delta']

    for col in needed.columns:
        needed[col] = pd.to_numeric(needed[col], errors='coerce')

    clean = needed.dropna(subset=['log_pO2', 'delta_atm_minus_delta', 'delta']).copy()

    sample_name = file_path.stem
    if sample_name.startswith('CT'):
        sample_name = sample_name[2:]

    print(f'{file_path.name}: {len(clean)} valid data point(s) loaded')

    if len(clean) == 0:
        print(f'Warning: {file_path.name} has no valid numeric rows after cleaning. Skipping in plots.')
        continue

    samples.append({'name': sample_name, 'data': clean})

if not samples:
    raise RuntimeError('No readable files with valid data were available for plotting.')

## 4) Plot 1: Column H vs Column I
Combined plot for all samples:
- x-axis: `log p(O2)` (Column H)
- y-axis: `delta at atm - delta` (Column I)

In [ ]:
plt.figure(figsize=(12, 8))
for sample in samples:
    d = sample['data']
    plt.plot(d['log_pO2'], d['delta_atm_minus_delta'], marker='o', linestyle='-', label=sample['name'])

plt.xlabel('log p(O2)')
plt.ylabel('delta at atm - delta')
plt.title('Coulometric Titration: log p(O2) vs delta at atm - delta')
plt.grid(True, alpha=0.3)
plt.legend(title='Sample', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 5) Plot 2: Column H vs Column J (inverted y-axis)
Combined plot for all samples:
- x-axis: `log p(O2)` (Column H)
- y-axis: `delta` (Column J)

The y-axis is inverted so higher `delta` values appear at the bottom.

In [ ]:
plt.figure(figsize=(12, 8))
for sample in samples:
    d = sample['data']
    plt.plot(d['log_pO2'], d['delta'], marker='o', linestyle='-', label=sample['name'])

plt.xlabel('log p(O2)')
plt.ylabel('delta')
plt.title('Coulometric Titration: log p(O2) vs delta')
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3)
plt.legend(title='Sample', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()